# LLMs as Music Code Generators

Large language models can generate executable music code. This transforms the creative process:
- Describe music in natural language → get working code
- Iterate via conversation: 'make the bass heavier', 'add swing'
- Bridge between musical intent and technical implementation

In this notebook we demonstrate LLM-style Python music code generation.

In [ ]:
!pip install pretty_midi matplotlib numpy IPython

In [ ]:
import pretty_midi
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio
import random

## Example 1: Generative Ambient Piece

This is the kind of code an LLM produces when asked:
*'Write Python code to create a generative ambient piece with evolving textures'*

In [ ]:
def generate_ambient(duration=30.0, seed=42):
    """Generate a generative ambient piece with pentatonic melodies
    and evolving dynamics."""
    random.seed(seed)
    np.random.seed(seed)

    midi = pretty_midi.PrettyMIDI(initial_tempo=72)

    # Pad: slow evolving chords
    pad = pretty_midi.Instrument(program=89, name='Pad')  # GM: Pad 2 (warm)

    # C pentatonic across octaves
    scale = [60, 62, 64, 67, 69,   # C4 pentatonic
             72, 74, 76, 79, 81]    # C5 pentatonic

    # Weighted random — favor root and fifth
    weights = [3, 1, 2, 2, 1, 3, 1, 2, 2, 1]
    weights = np.array(weights, dtype=float)
    weights /= weights.sum()

    t = 0.0
    while t < duration:
        # Choose 2-3 notes for a cluster
        n_notes = random.choice([2, 2, 3])
        chosen = np.random.choice(scale, size=n_notes, replace=False, p=weights)

        # Varying note durations (2-6 seconds)
        note_dur = random.uniform(2.0, 6.0)

        # Varying dynamics
        base_vel = random.randint(40, 80)

        for pitch in chosen:
            vel = min(127, base_vel + random.randint(-10, 10))
            note = pretty_midi.Note(
                velocity=vel,
                pitch=int(pitch),
                start=t,
                end=min(t + note_dur, duration)
            )
            pad.notes.append(note)

        t += random.uniform(1.5, 4.0)  # Overlap between clusters

    # Melody: sparse high notes
    melody = pretty_midi.Instrument(program=46, name='Melody')  # GM: Harp
    high_scale = [79, 81, 84, 86, 88, 91, 93]  # G5-A6 pentatonic

    t = 2.0  # Start after pad
    while t < duration:
        pitch = random.choice(high_scale)
        vel = random.randint(50, 90)
        note_dur = random.uniform(0.5, 2.0)
        note = pretty_midi.Note(
            velocity=vel, pitch=pitch,
            start=t, end=min(t + note_dur, duration)
        )
        melody.notes.append(note)
        t += random.uniform(1.0, 5.0)  # Sparse timing

    midi.instruments.append(pad)
    midi.instruments.append(melody)

    return midi


# Generate and save
ambient = generate_ambient(duration=30.0)
ambient.write('ambient_output.mid')
print(f"Generated {len(ambient.instruments)} instruments")
for inst in ambient.instruments:
    print(f"  {inst.name}: {len(inst.notes)} notes")

# Synthesize and play
audio = ambient.fluidsynth(fs=22050)
Audio(audio, rate=22050)

# Piano roll visualization
fig, ax = plt.subplots(figsize=(14, 4))
for inst in ambient.instruments:
    for note in inst.notes:
        ax.barh(note.pitch, note.end - note.start, left=note.start,
                height=0.7, alpha=0.7,
                color='steelblue' if inst.name == 'Pad' else 'coral')
ax.set_xlabel('Time (s)', fontsize=12)
ax.set_ylabel('MIDI Pitch', fontsize=12)
ax.set_title('Generative Ambient — Piano Roll', fontsize=14)
ax.legend(['Pad', 'Melody'])
plt.tight_layout()
plt.show()

## Example 2: Walking Bass Line

Prompt: *'Write a jazz walking bass in Bb at 120 BPM with chord tones on beats 1 and 3, approach notes on 2 and 4'*

In [ ]:
def generate_walking_bass(bars=8, tempo=120):
    """Generate a jazz walking bass line over a Bb blues progression.
    Chord tones on beats 1 and 3, chromatic approaches on 2 and 4."""
    midi = pretty_midi.PrettyMIDI(initial_tempo=tempo)
    bass = pretty_midi.Instrument(program=32, name='Acoustic Bass')  # GM: Acoustic Bass

    beat_dur = 60.0 / tempo  # Duration of one beat

    # Bb blues changes (each chord = 1 bar = 4 beats)
    # Bb7 | Eb7 | Bb7 | Bb7 | Eb7 | Eb7 | Bb7 | F7
    changes = [
        {'root': 46, 'tones': [46, 50, 53, 56]},  # Bb7: Bb Db F Ab
        {'root': 51, 'tones': [51, 55, 58, 61]},  # Eb7: Eb Gb Bb Db
        {'root': 46, 'tones': [46, 50, 53, 56]},  # Bb7
        {'root': 46, 'tones': [46, 50, 53, 56]},  # Bb7
        {'root': 51, 'tones': [51, 55, 58, 61]},  # Eb7
        {'root': 51, 'tones': [51, 55, 58, 61]},  # Eb7
        {'root': 46, 'tones': [46, 50, 53, 56]},  # Bb7
        {'root': 53, 'tones': [53, 57, 60, 63]},  # F7: F A C Eb
    ]

    t = 0.0
    for bar_idx in range(min(bars, len(changes))):
        chord = changes[bar_idx]
        next_chord = changes[(bar_idx + 1) % len(changes)]

        for beat in range(4):
            if beat == 0:
                # Beat 1: root
                pitch = chord['root']
            elif beat == 2:
                # Beat 3: chord tone (5th or 3rd)
                pitch = random.choice(chord['tones'][1:3])
            elif beat == 1:
                # Beat 2: chromatic approach to beat 3 target
                target = random.choice(chord['tones'][1:3])
                pitch = target + random.choice([-1, 1])
            else:
                # Beat 4: chromatic approach to next bar's root
                pitch = next_chord['root'] + random.choice([-1, 1])

            # Keep in bass range (Bb1 to Bb3)
            while pitch < 34:
                pitch += 12
            while pitch > 58:
                pitch -= 12

            vel = 90 if beat in [0, 2] else 75  # Accent beats 1 and 3
            note = pretty_midi.Note(
                velocity=vel, pitch=pitch,
                start=t, end=t + beat_dur * 0.9
            )
            bass.notes.append(note)
            t += beat_dur

    midi.instruments.append(bass)
    return midi


# Generate
random.seed(7)
bass_midi = generate_walking_bass(bars=8, tempo=120)
bass_midi.write('walking_bass.mid')

print("Walking bass line generated:")
for note in bass_midi.instruments[0].notes:
    name = pretty_midi.note_number_to_name(note.pitch)
    beat_num = int(note.start / (60.0 / 120)) % 4 + 1
    bar_num = int(note.start / (60.0 / 120)) // 4 + 1
    print(f"  Bar {bar_num}, Beat {beat_num}: {name} (vel={note.velocity})")

# Synthesize and play
audio = bass_midi.fluidsynth(fs=22050)
Audio(audio, rate=22050)

## Example 3: Drum Pattern Generator

Prompt: *'Create a drum pattern generator that supports different styles: rock, jazz, bossa nova'*

In [ ]:
def generate_drums(style='rock', bars=4, tempo=120):
    """Generate drum patterns in various styles using GM drum map."""
    midi = pretty_midi.PrettyMIDI(initial_tempo=tempo)
    drums = pretty_midi.Instrument(program=0, is_drum=True, name='Drums')

    beat_dur = 60.0 / tempo
    eighth = beat_dur / 2
    total_beats = bars * 4

    # GM drum map
    KICK = 36
    SNARE = 38
    CLOSED_HH = 42
    OPEN_HH = 46
    RIDE = 51
    CRASH = 49
    LOW_TOM = 45
    HI_TOM = 50
    SIDE_STICK = 37

    # Style definitions: list of (pitch, time_offsets_per_bar, velocity)
    patterns = {
        'rock': {
            KICK:      {'times': [0, 2],                 'vel': 100},
            SNARE:     {'times': [1, 3],                 'vel': 95},
            CLOSED_HH: {'times': [i * 0.5 for i in range(8)], 'vel': 70},
        },
        'jazz': {
            RIDE:       {'times': [0, 0.67, 1, 1.67, 2, 2.67, 3, 3.67], 'vel': 75},
            KICK:       {'times': [0, 2.5],              'vel': 60},
            CLOSED_HH:  {'times': [1, 3],                'vel': 50},
        },
        'bossa_nova': {
            SIDE_STICK: {'times': [0.5, 1.25, 2.5, 3.25], 'vel': 80},
            KICK:       {'times': [0, 1.5, 3],           'vel': 75},
            CLOSED_HH:  {'times': [i * 0.5 for i in range(8)], 'vel': 55},
        },
    }

    if style not in patterns:
        raise ValueError(f"Unknown style: {style}. Choose from {list(patterns.keys())}")

    pattern = patterns[style]

    for bar in range(bars):
        bar_start = bar * 4 * beat_dur
        for pitch, info in pattern.items():
            for beat_offset in info['times']:
                t = bar_start + beat_offset * beat_dur
                vel = info['vel'] + random.randint(-5, 5)  # Humanize
                vel = max(1, min(127, vel))
                t += random.uniform(-0.01, 0.01)  # Timing humanization
                note = pretty_midi.Note(
                    velocity=vel, pitch=pitch,
                    start=max(0, t), end=max(0, t) + 0.1
                )
                drums.notes.append(note)

    midi.instruments.append(drums)
    return midi


# Generate all three styles and display
fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)

for idx, style in enumerate(['rock', 'jazz', 'bossa_nova']):
    random.seed(42)
    drum_midi = generate_drums(style=style, bars=4, tempo=120)
    drum_midi.write(f'drums_{style}.mid')

    # Plot
    ax = axes[idx]
    for note in drum_midi.instruments[0].notes:
        ax.scatter(note.start, note.pitch, s=note.velocity * 2,
                   alpha=0.7, color=['steelblue', 'coral', 'seagreen'][idx])
    ax.set_ylabel('GM Drum #')
    ax.set_title(f'{style.replace("_", " ").title()} Pattern', fontsize=12)

axes[-1].set_xlabel('Time (s)', fontsize=12)
plt.suptitle('Drum Patterns: Rock, Jazz, Bossa Nova', fontsize=14)
plt.tight_layout()
plt.show()

# Play the rock pattern
rock_audio = generate_drums('rock', bars=4, tempo=120).fluidsynth(fs=22050)
Audio(rock_audio, rate=22050)

## Prompt Engineering for Music Code

Better prompts produce better code:
- **Vague**: 'Make music' → generic, often broken
- **Specific**: Include key, tempo, time signature, instruments, style
- **Constrained**: Add musical rules (voice leading, range limits, rhythmic patterns)

In [ ]:
# Prompt comparison examples
prompts = [
    {
        'quality': 'Vague',
        'prompt': 'Write code to make music',
        'issues': 'No key, tempo, style, or structure specified. Output is unpredictable.',
    },
    {
        'quality': 'Better',
        'prompt': 'Write Python code using pretty_midi to create a 16-bar jazz piano piece in C minor at 140 BPM',
        'issues': 'Specifies tool, genre, key, tempo, length. But no harmonic details.',
    },
    {
        'quality': 'Best',
        'prompt': ('Write Python code using pretty_midi to create a 16-bar jazz piano piece. '
                   'Key: C minor. Tempo: 140 BPM. Use ii-V-I progressions with rootless voicings. '
                   'Left hand plays shell voicings (root + 7th). Right hand plays chord tones with '
                   'occasional chromatic passing tones. Swing eighth notes (2:1 ratio). '
                   'Dynamics: mp to mf, crescendo into each phrase peak.'),
        'issues': 'Specifies harmonic language, voicing style, rhythm feel, and dynamics.',
    },
]

print("Prompt Engineering for Music Code Generation")
print("=" * 60)
for p in prompts:
    print(f"\n[{p['quality']}]")
    print(f"  Prompt: \"{p['prompt']}\"")
    print(f"  Why:    {p['issues']}")

## SuperCollider Code Examples

LLMs can also generate SuperCollider code (sent via OSC bridge):

In [ ]:
# SuperCollider code that an LLM might generate
# These are strings — to execute them, send via OSC to SC (see claude_osc_bridge.scd)

sc_examples = {
    'Pbind pattern (arpeggiated synth)': '''
// Arpeggiated pattern in Dorian mode
Pdef(\\arp, Pbind(
    \\instrument, \\default,
    \\scale, Scale.dorian,
    \\degree, Pseq([0, 2, 4, 6, 7, 6, 4, 2], inf),
    \\octave, Prand([4, 5, 5, 6], inf),
    \\dur, 0.125,
    \\amp, Pwhite(0.05, 0.15),
    \\sustain, 0.3,
)).play;
''',

    'SynthDef (analog pad)': '''
// Warm analog pad with detuned oscillators
SynthDef(\\warmPad, { |out=0, freq=220, amp=0.3, gate=1, detune=0.5|
    var sig, env;
    sig = Mix.fill(4, { |i|
        Saw.ar(freq * (1 + (i * detune * 0.01))) * 0.25
    });
    sig = MoogFF.ar(sig, freq * 3 + (LFTri.kr(0.1) * freq), 2.5);
    env = EnvGen.kr(Env.adsr(0.5, 0.3, 0.7, 1.0), gate, doneAction: 2);
    Out.ar(out, Pan2.ar(sig * env * amp, 0));
}).add;
''',

    'Pdef (evolving generative pattern)': '''
// Evolving generative pattern with random walks
Pdef(\\evolve, Pbind(
    \\instrument, \\default,
    \\scale, Scale.minorPentatonic,
    \\degree, Pbrown(-7, 14, 2),  // random walk
    \\dur, Pwrand([0.25, 0.5, 1.0], [0.5, 0.3, 0.2], inf),
    \\amp, Pwhite(0.05, 0.2),
    \\pan, Pbrown(-0.8, 0.8, 0.2),
)).play;
''',
}

print("SuperCollider Code Examples (LLM-generated)")
print("=" * 60)
print("Send these to SC via: client.send_message('/claude/eval', code)")
print()
for title, code in sc_examples.items():
    print(f"--- {title} ---")
    print(code)